# VibeVoice Pipeline on Colab

Runs the full Video Generation Pipeline on Colab's free GPU (T4).


## 0. Clone repositories


In [ ]:
# ── GPU check ────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), (
    "No GPU! Top menu: Runtime -> Change runtime type -> T4 GPU, then re-run all cells. "
    "On CPU the model load exhausts RAM and Colab kills the process."
)
print("GPU:", torch.cuda.get_device_name(0))

# ── Model choice ─────────────────────────────────────────────
# "0.5B"  — fast streaming model, fixed .pt voices (free T4 OK)
# "1.5B"  — TRUE voice cloning from a reference WAV (free T4 OK, ~7GB)
# "Large" — the 7B-class model, best quality (needs Colab Pro A100/L4, ~20GB VRAM)
VIBEVOICE_MODEL = "1.5B"

import os
os.environ["VIBEVOICE_MODEL"] = VIBEVOICE_MODEL

!rm -rf audio
!git clone https://github.com/saitejatiru/audiowithvideoremotion.git audio
%cd audio
!rm -rf VibeVoice
if VIBEVOICE_MODEL == "0.5B":
    # microsoft repo has the streaming model code + .pt voices
    !git clone https://github.com/microsoft/VibeVoice.git
else:
    # community fork restores the non-streaming TTS code + reference WAV voices
    !git clone https://github.com/vibevoice-community/VibeVoice.git
import sys; sys.path.insert(0, ".")


## 1. Install dependencies
Installs requirements for TTS, Alignment, Storyboarding, Rendering, and Orchestration.


In [ ]:
!pip install -q fastapi "uvicorn[standard]" soundfile requests nemo-text-processing gradio tenacity "pandas<3.0.0" "huggingface-hub>=1.2.0"
!pip install -q -r align/requirements.txt
!pip install -q openai json-repair

# VibeVoice model package — REQUIRED, run_vibevoice imports `vibevoice`
!pip install -q -e VibeVoice

# Node.js + Remotion dependencies for Phase 4 rendering
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!cd video && npm install
!sudo apt-get install -yq gconf-service libasound2 libatk1.0-0 libc6 libcairo2 libcups2 libdbus-1-3 libexpat1 libfontconfig1 libgcc1 libgconf-2-4 libgdk-pixbuf2.0-0 libglib2.0-0 libgtk-3-0 libnspr4 libpango-1.0-0 libpangocairo-1.0-0 libstdc++6 libx11-6 libx11-xcb1 libxcb1 libxcomposite1 libxcursor1 libxdamage1 libxext6 libxfixes3 libxi6 libxrandr2 libxrender1 libxss1 libxtst6 ca-certificates fonts-liberation libappindicator1 libnss3 lsb-release xdg-utils wget

## 2. Set API Keys


In [ ]:
import os

# Kimi K2.6 via NVIDIA NIM. Key from build.nvidia.com (starts with "nvapi-").
# NIM model ids are "org/model" — copy the EXACT id from the code sample on
# the model's page at build.nvidia.com. If the smoke test below fails, it
# prints the available ids — copy the right one from there.
os.environ["LLM_BASE_URL"] = "https://integrate.api.nvidia.com/v1"
os.environ["LLM_API_KEY"]  = "<YOUR_NVAPI_KEY>"
os.environ["LLM_MODEL"]    = "moonshotai/kimi-k2.6"

# Alternative — DeepSeek (verified with our JSON mode):
# os.environ["LLM_BASE_URL"] = "https://api.deepseek.com"
# os.environ["LLM_API_KEY"]  = "sk-..."
# os.environ["LLM_MODEL"]    = "deepseek-chat"

if "<" in os.environ["LLM_API_KEY"]:
    print("*" * 70)
    print("WARNING: LLM_API_KEY is still a placeholder!")
    print("Without a real key the storyboard falls back to PLAIN TEXT scenes")
    print("(no charts, no diagrams, no animations). Paste a real key above.")
    print("*" * 70)
else:
    # Smoke-test the LLM config NOW instead of failing silently mid-pipeline
    from openai import OpenAI
    _c = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ["LLM_BASE_URL"])
    try:
        _r = _c.chat.completions.create(
            model=os.environ["LLM_MODEL"],
            messages=[{"role": "user", "content": "Reply with the word: ok"}],
            max_tokens=10,
        )
        print("LLM OK:", os.environ["LLM_MODEL"], "->", _r.choices[0].message.content)
    except Exception as e:
        print("*" * 70)
        print("LLM CONFIG BROKEN — storyboard would fall back to plain text scenes!")
        print("Error:", e)
        try:
            _ids = [m.id for m in _c.models.list().data]
            _kimi = [i for i in _ids if "kimi" in i.lower()] or _ids[:20]
            print("Available model ids (pick the right one):", _kimi)
        except Exception:
            pass
        print("Fix LLM_MODEL above. DO NOT generate videos until this prints 'LLM OK'.")
        print("*" * 70)

# Enable Gradio Share link
os.environ["GRADIO_SHARE"] = "1"


## 3. Launch the Full Gradio UI
This will expose the complete pipeline (TTS -> Align -> Storyboard -> Render -> Post-Process) through a public Gradio URL.


In [ ]:
%cd /content/audio
!python app.py
